In [12]:
import lightgbm
import mlflow
import os
from dotenv import load_dotenv
import sys
import pandas as pd
import json
from sklearn.inspection import permutation_importance
import numpy as np

In [2]:
pd.set_option("display.max_columns", 100)
load_dotenv()
data_path = os.getenv("DATA_PATH")
src_path = os.getenv("SRC_PATH")
sys.path.append(src_path)
json_path = os.path.join(data_path, "processed/split_info.json")
with open(json_path) as f:
    json_info = json.load(f)
train_end = json_info.get("train_end")
val_end = json_info.get("validation_end")
from about_data.data_load import load_df
from about_data.split import temporal_split
from features.engineering import create_d_features, create_advanced_time_features, add_distance_features, add_interaction_features, add_all_features
from model.preprocessor_pipe_evalueate import create_pipeline, evaluate_model, get_preprocessor

full_df = load_df(data_path)
train, val, test = temporal_split(full_df, train_end, val_end)

In [3]:
map_dfs = {"train": train, "val": val}

x_dfs = {}
y_dfs = {}

for name, sample_df in map_dfs.items():
    x_dfs[name] = add_all_features(sample_df)
    y_dfs[name] = sample_df['isFraud']

In [4]:
y_dfs['train'].shape, x_dfs['train'].shape, y_dfs['val'].shape, x_dfs['val'].shape  

((413378,), (413378, 48), (88581,), (88581, 48))

In [5]:
model = lightgbm.LGBMClassifier(n_estimators=300, learning_rate=0.05, objective="binary", metric='average_precision', random_state=42, n_jobs=-1, importance_type="gain")

pipe = create_pipeline(model, get_preprocessor(x_dfs['train']))
pipe.fit(x_dfs['train'], y_dfs['train'])
metrics = evaluate_model(pipe, x_dfs['val'], y_dfs['val'])

[LightGBM] [Info] Number of positive: 14538, number of negative: 398840
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.087782 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 13884
[LightGBM] [Info] Number of data points in the train set: 413378, number of used features: 5458
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035169 -> initscore=-3.311794
[LightGBM] [Info] Start training from score -3.311794


In [6]:
metrics

{'pr_auc': 0.37556114184645,
 'roc_auc': 0.8796284478901586,
 'precision': 0.6651305683563749,
 'recall': 0.1423405654174885,
 'f1': 0.23449769834822637}

In [7]:
perm = permutation_importance(pipe, x_dfs['val'], y_dfs['val'], scoring = "average_precision", n_repeats=5, random_state=42, n_jobs=-1)

In [8]:
importance_df = pd.DataFrame({"feature": x_dfs['val'].columns, "importance_mean": perm.importances_mean, "importance_std": perm.importances_std }).sort_values("importance_mean", ascending=False)

In [9]:
importance_df

,feature,importance_mean,importance_std
2,ProductCD,0.073269,0.001080
17,missing_D_count,0.064620,0.001590
39,card_addr,0.060987,0.001354
14,D1,0.060212,0.001133
27,amount_decimal,0.048365,0.003582
9,card6,0.037721,0.001539
15,missing_count,0.031183,0.001419
3,P_emaildomain,0.030676,0.001337
26,transaction_amt_log,0.024870,0.001990
20,missing_id_count,0.024294,0.001593


In [43]:
feature_count = list(range(10, 41, 6)) + [48]
features = x_dfs['val'].columns.tolist()

results = []

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("feature-selection")

for n in feature_count:
    with mlflow.start_run(run_name=f"{n}_features"):

        selected = features[:n]
    
        X_train_selected = x_dfs['train'][selected]
        X_val_selected = x_dfs['val'][selected]
    
        pipe = create_pipeline(model, get_preprocessor(X_val_selected))
        pipe.fit(X_train_selected, y_dfs['train'])
        metrics = evaluate_model(pipe, X_val_selected, y_dfs['val'])
    
        mlflow.log_param("number_of_features",  n)
        mlflow.log_param("model", "lightgbm")
        mlflow.log_metrics(metrics)
    
        results.append({"feature_count": n, **metrics})

[LightGBM] [Info] Number of positive: 14538, number of negative: 398840
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.053686 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1326
[LightGBM] [Info] Number of data points in the train set: 413378, number of used features: 129
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035169 -> initscore=-3.311794
[LightGBM] [Info] Start training from score -3.311794
🏃 View run 10_features at: http://127.0.0.1:5000/#/experiments/5/runs/c10ad0d3897d41fcad9adafe79dc0a1f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
[LightGBM] [Info] Number of positive: 14538, number of negative: 398840
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.056497 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `

In [44]:
pd.DataFrame(results).sort_values("pr_auc", ascending=False)

,feature_count,pr_auc,roc_auc,precision,recall,f1
6,48,0.375561,0.879628,0.665131,0.142341,0.234498
5,40,0.372754,0.878931,0.672673,0.147272,0.241640
3,28,0.350555,0.866865,0.684211,0.141026,0.233851
2,22,0.344565,0.868639,0.663609,0.142669,0.234848
4,34,0.343280,0.864704,0.670827,0.141354,0.233505
1,16,0.318864,0.858491,0.660305,0.113741,0.194055
0,10,0.248582,0.816314,0.662602,0.053583,0.099148
